In [1]:
import os
import re
import math
import heapq
import random
import faiss
from typing import List,Tuple, Optional
import numpy as np
import plotly.graph_objects as go
from sklearn.cluster import KMeans
import time
from scipy.io import loadmat


In [2]:
# ---------------------
# Pure-Python KHNSW implementation 
# ---------------------
class Node:
    def __init__(self, idx: int, vector: np.ndarray, level: int):
        self.idx = idx
        self.vector = vector
        self.level = level
        # neighbors[l] exists for 0..level inclusive
        self.neighbors: List[List[int]] = [list() for _ in range(level + 1)]
        self.deleted = False

class HNSW:
    def __init__(self, M=16, ef_construction=200, random_seed=None):
        if random_seed is not None:
            random.seed(random_seed)
            np.random.seed(random_seed)

        self.M = M
        self.ef_construction = ef_construction
        self.level_mult = 1.0 / math.log(M) if M > 1 else 1.0
        self.entry_point: Optional[int] = None
        self.max_level: int = -1
        self.nodes: List[Optional[Node]] = []
        # visited bookkeeping
        self._visit_mark = 1
        self._visited = {}

    @staticmethod
    def _distance(a: np.ndarray, b: np.ndarray) -> float:
        return float(np.linalg.norm(a - b))

    def _get_random_level(self) -> int:
        r = random.random()
        if r == 0.0:
            r = 1e-16
        return int(math.floor(-math.log(r) * self.level_mult))

    def _next_visit(self):
        self._visit_mark += 1
        if self._visit_mark % 1000000 == 0:
            self._visited.clear()

    def _is_visited(self, idx: int) -> bool:
        return self._visited.get(idx, 0) == self._visit_mark

    def _set_visited(self, idx: int):
        self._visited[idx] = self._visit_mark

    # -------------------------
    # Safe neighbor access: return neighbor list at level if node participates, else empty list
    # -------------------------
    def _neighbors_at_level(self, node_idx: int, level: int) -> List[int]:
        node = self.nodes[node_idx]
        if node is None:
            return []
        if level <= node.level:
            return node.neighbors[level]
        return []

    # -------------------------
    # Greedy search (safe)
    # -------------------------
    def _greedy_search_layer(self, query_vec: np.ndarray, entry_idx: int, level: int) -> int:
        current = entry_idx
        current_dist = self._distance(query_vec, self.nodes[current].vector)
        improved = True
        while improved:
            improved = False
            neighs = self._neighbors_at_level(current, level)
            for nb in neighs:
                d = self._distance(query_vec, self.nodes[nb].vector)
                if d < current_dist:
                    current = nb
                    current_dist = d
                    improved = True
        return current

    # -------------------------
    # Best-first (ef) search at a level (safe)
    # -------------------------
    def _search_layer(self, query_vec: np.ndarray, entry_idx: int, ef: int, level: int) -> List[Tuple[float, int]]:
        candidates: List[Tuple[float,int]] = []
        result_heap: List[Tuple[float,int]] = []  # store (-dist, idx)

        self._next_visit()
        entry_dist = self._distance(query_vec, self.nodes[entry_idx].vector)
        heapq.heappush(candidates, (entry_dist, entry_idx))
        heapq.heappush(result_heap, (-entry_dist, entry_idx))
        self._set_visited(entry_idx)

        while candidates:
            top_dist, top_idx = candidates[0]  # peek min
            worst_dist = -result_heap[0][0]    # max in result_heap
            if top_dist > worst_dist and len(result_heap) >= ef:
                break
            heapq.heappop(candidates)

            neighs = self._neighbors_at_level(top_idx, level)
            for nb in neighs:
                if self._is_visited(nb):
                    continue
                self._set_visited(nb)
                d = self._distance(query_vec, self.nodes[nb].vector)
                if len(result_heap) < ef or d < -result_heap[0][0]:
                    heapq.heappush(candidates, (d, nb))
                    heapq.heappush(result_heap, (-d, nb))
                    if len(result_heap) > ef:
                        heapq.heappop(result_heap)

        res = [(-dist, idx) for (dist, idx) in result_heap]
        res.sort(key=lambda x: x[0])
        return res

    # -------------------------
    # Neighbor selection heuristic (paper)
    # -------------------------
    def _select_neighbors(self, query_vec: np.ndarray, candidates: List[Tuple[float,int]], M: int) -> List[int]:
        selected: List[int] = []
        for (dist_q, cand_idx) in candidates:
            cand_vec = self.nodes[cand_idx].vector
            ok = True
            for sel_idx in selected:
                if self._distance(cand_vec, self.nodes[sel_idx].vector) < dist_q:
                    ok = False
                    break
            if ok:
                selected.append(cand_idx)
            if len(selected) >= M:
                break
        return selected

    def _select_neighbors_from_ids(self, query_vec: np.ndarray, ids: List[int], M: int) -> List[int]:
        cand_with_dist = [(self._distance(query_vec, self.nodes[idx].vector), idx) for idx in ids]
        cand_with_dist.sort(key=lambda x: x[0])
        return self._select_neighbors(query_vec, cand_with_dist, M)

    # -------------------------
    # Add item (supports level_override)
    # -------------------------
    def add_item(self, vector: np.ndarray, idx: Optional[int] = None, level_override: Optional[int] = None):
        if idx is None:
            idx = len(self.nodes)
        vector = np.asarray(vector, dtype=float)

        # choose level
        if level_override is None:
            level = self._get_random_level()
        else:
            level = level_override

        node = Node(idx, vector, level)
        # ensure nodes array is big enough
        if idx >= len(self.nodes):
            self.nodes.extend([None] * (idx - len(self.nodes) + 1))
        self.nodes[idx] = node

        # first node
        if self.entry_point is None:
            self.entry_point = idx
            self.max_level = level
            return

        ep = self.entry_point

        # greedy descent on layers above new node's level
        for L in range(self.max_level, level, -1):
            ep = self._greedy_search_layer(vector, ep, L)

        # for layers from min(level, max_level) down to 0, search and link
        for L in range(min(level, self.max_level), -1, -1):
            # find candidates using ef_construction
            candidates = self._search_layer(vector, ep, self.ef_construction, L)
            selected = self._select_neighbors(vector, candidates, self.M)
            node.neighbors[L] = selected.copy()

            # link back: append new node to each neighbor's adjacency at level L
            for nb in selected:
                nb_node = self.nodes[nb]
                # defensive: extend nb_node.neighbors list if it doesn't have level L
                if L > nb_node.level:
                    # extend neighbor lists to include L (empty lists for new levels)
                    nb_node.neighbors.extend([[] for _ in range(L - nb_node.level)])
                    nb_node.level = L
                nb_node.neighbors[L].append(idx)

                # if neighbor list too long, prune using heuristic with neighbor as query
                if len(nb_node.neighbors[L]) > self.M:
                    pruned = self._select_neighbors_from_ids(nb_node.vector, nb_node.neighbors[L], self.M)
                    nb_node.neighbors[L] = pruned

            # update ep for next lower layer to the closest candidate
            if candidates:
                ep = candidates[0][1]

        # if node has higher level than current max, update entry point
        if level > self.max_level:
            self.entry_point = idx
            self.max_level = level

    # -------------------------
    # k-NN search (public)
    # -------------------------
    def search_knn(self, query_vec: np.ndarray, k: int = 1, ef_search: Optional[int] = None) -> List[Tuple[float,int]]:
        if self.entry_point is None:
            return []
        if ef_search is None:
            ef_search = max(k, 50)

        ep = self.entry_point
        # greedy descent through higher layers
        for L in range(self.max_level, 0, -1):
            ep = self._greedy_search_layer(query_vec, ep, L)

        # ef_search on level 0
        results = self._search_layer(query_vec, ep, ef_search, 0)
        return results[:k]

In [3]:
# ---------------------
# Leader selection via KMeans
# ---------------------
def choose_k_leaders_by_kmeans(X_index: np.ndarray, k: int, random_seed: int = 42) -> List[int]:
    if k <= 0:
        return []

    kmeans = KMeans(n_clusters=k, random_state=random_seed)
    kmeans.fit(X_index)
    centers = kmeans.cluster_centers_

    leaders: List[int] = []
    assigned = set()
    for c in centers:
        dists = np.linalg.norm(X_index - c, axis=1)
        for idx in np.argsort(dists):
            if idx not in assigned:
                leaders.append(int(idx))
                assigned.add(idx)
                break
    return leaders

# ---------------------
# Build HNSW with leaders forced to top level
# ---------------------
def build_hnsw_with_leaders(X_index: np.ndarray, k_leaders: int, M=16, ef_construction=200, random_seed=42) -> Tuple[HNSW, List[int]]:
    n = X_index.shape[0]
    if k_leaders <= 0 or k_leaders >= n:
        raise ValueError("k_leaders must be >0 and < number of index vectors")

    hnsw = HNSW(M=M, ef_construction=ef_construction, random_seed=random_seed)

    # choose leaders
    leaders = choose_k_leaders_by_kmeans(X_index, k_leaders, random_seed)
    print(f"Chosen {len(leaders)} leaders (indices): {leaders}")

    # sample levels and set a top_level for leaders (gives them high layer)
    sampled_levels = [hnsw._get_random_level() for _ in range(max(50, k_leaders*5))]
    top_level = max(sampled_levels)
    if top_level < 1:
        top_level = 1
    print(f"Forcing leaders to level {top_level} (sampled levels example: {sampled_levels[:10]})")

    # Add leaders first, with level_override
    for li in leaders:
        hnsw.add_item(X_index[li], idx=li, level_override=top_level)

    # Add remaining vectors
    for i in range(n):
        if i in leaders:
            continue
        hnsw.add_item(X_index[i], idx=i)

    # ensure entry_point is leader (nice to have)
    if hnsw.entry_point not in leaders:
        hnsw.entry_point = leaders[0]
        hnsw.max_level = max(hnsw.max_level, top_level)

    return hnsw, leaders

In [18]:
from tqdm import tqdm
import numpy as np

def evaluate_hit_rate_topk(hnsw_index, index_labels, query_vectors, query_labels, k_max=50, ef_search=20):
    hit_rates = []
    x_labels = []
    total_indexed = len(index_labels)

    if total_indexed == 0 or len(query_labels) == 0:
        raise ValueError("Index or query set empty.")

    k_max = min(k_max, total_indexed)

    # Outer loop over k values with tqdm
    for k in tqdm(range(1, k_max + 1,10), desc="Evaluating top-K hit rates", position=0):
        hits = 0

        # Inner loop over queries with tqdm
        for qvec, qlabel in tqdm(zip(query_vectors, query_labels),
                                 total=len(query_labels),
                                 desc=f"K={k}",
                                 leave=False,
                                 position=1):
            res = hnsw_index.search_knn(qvec, k=k, ef_search=ef_search)
            if len(res) == 0:
                continue
            retrieved_idxs = [item[1] for item in res]
            retrieved_labels = index_labels[retrieved_idxs]
            if np.any(retrieved_labels == qlabel):
                hits += 1

        hit_rate = hits / len(query_labels)
        hit_rates.append(hit_rate)
        penetration = k / total_indexed
        x_labels.append(f"{k} ({penetration:.3f})")

        tqdm.write(f"K={k:3d}, Hit rate={hit_rate:.4f}")

    return x_labels, hit_rates


In [5]:
# =========================================================
#  Plot Hit Rate vs Penetration Rate
# =========================================================
def plot_hit_rate_plotly_topk(penetration_rates, hit_rates, efSearch,title_name="IITD_HNSW"):

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=penetration_rates,
        y=hit_rates,
        mode='lines+markers',
        name=f"efSearch={efSearch}",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    # numeric_vals = [
    #     float(re.search(r'\((.*?)\)', str(x)).group(1)) if '(' in str(x) else float(x)
    #     for x in penetration_rates
    # ]
    # max_tick = math.ceil(max(numeric_vals) * 100) / 100  # round up to next 0.01
    # # Clean tick marks at 0.01 intervals
    # tick_vals = np.round(np.arange(0.0, max_tick + 0.001, 0.01), 2).tolist()
    fig.update_xaxes(
        title='Penetration Rate (k / total indexed)',
        # tickvals=tick_vals,
        # tickformat=".2f"
    )
    fig.update_yaxes(title='Hit Rate')

    fig.update_layout(
        title=f"{title_name}",
        template='plotly_white',
        width=800,
        height=500,
        hovermode="x unified"
    )
    fig.show()


In [6]:
import time
import numpy as np
from typing import Tuple, List, Dict

def evaluate_top1_timing(hnsw_index,
                         index_labels: np.ndarray,
                         query_vectors: np.ndarray,
                         query_labels: np.ndarray,
                         ef_search: int = 20) -> Dict[str, object]:
    """
    For each query vector:
      - perform a top-1 search (k=1)
      - measure the time taken for the search call
      - check if the retrieved top-1 label matches the query label

    Returns a dict containing:
      - 'hit_rate': fraction of queries whose top-1 label matched
      - 'avg_time_s': average search time per query in seconds
      - 'times_s': list of per-query times (seconds) in the same order as query_vectors
      - 'total_queries': number of queries
      - 'successful_searches': number of searches that returned at least one neighbor
    """
    if len(index_labels) == 0 or len(query_labels) == 0:
        raise ValueError("Index or query set empty.")

    if len(query_vectors) != len(query_labels):
        raise ValueError("query_vectors and query_labels must have the same length.")

    times: List[float] = []
    hits = 0
    successful_searches = 0
    total_queries = len(query_labels)

    for qvec, qlabel in zip(query_vectors, query_labels):
        t0 = time.perf_counter()
        res = hnsw_index.search_knn(qvec, k=1, ef_search=ef_search)
        t1 = time.perf_counter()

        elapsed = t1 - t0
        times.append(elapsed)

        # If the index returned at least one neighbor, check label
        if len(res) > 0:
            successful_searches += 1
            # assume res is iterable of (distance, index) or similar
            # take the index of the first (top-1) result
            top_idx = res[0][1]
            if index_labels[top_idx] == qlabel:
                hits += 1

    hit_rate = hits / total_queries
    avg_time_s = float(sum(times) / total_queries) if total_queries > 0 else 0.0

    return {
        "hit_rate": hit_rate,
        "avg_time_s": avg_time_s,
        "times_s": times,
        "total_queries": total_queries,
        "successful_searches": successful_searches
    }


# IITDV1

In [7]:
# ---------------------
# Load IITD_UPD Templates (70:30 Split) - as provided by you
# ---------------------
def load_iitd_upd_templates(root_folder, seed=42, split_ratio=0.7):
    """
    For each subject folder (like 1_L, 2_L...), randomly split its .npy files
      - 70% for indexing (gallery/enrollment)
      - 30% for query (probe)
    Returns:
        index_vectors, index_labels, query_vectors, query_labels
    """
    random.seed(seed)
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    subject_folders = sorted(
        [d for d in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, d))]
    )

    for subject in subject_folders:
        subject_path = os.path.join(root_folder, subject)
        files = sorted([f for f in os.listdir(subject_path) if f.endswith('.npy')])

        n = len(files)
        if n == 0:
            print(f"⚠️ Skipping {subject}, no .npy files found.")
            continue

        random.shuffle(files)
        split_point = max(1, int(n * split_ratio))  # At least 1 for indexing
        index_files = files[:split_point]
        query_files = files[split_point:] if split_point < n else []

        # Load index vectors
        for f in index_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            index_vectors.append(vec)
            index_labels.append(subject)

        # Load query vectors
        for f in query_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            query_vectors.append(vec)
            query_labels.append(subject)

        print(f"{subject}: {len(index_files)} index, {len(query_files)} query")

    if len(index_vectors) == 0:
        raise ValueError("No index vectors found. Check `root_folder` and structure.")

    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors) if len(query_vectors) > 0 else np.zeros((0, index_vectors[0].shape[0]), dtype='float32'),
        np.array(query_labels)
    )


In [8]:
data_folder = "/home/nishkal/sg/iris_indexing/IITD_UPD"
X_index, y_index, X_query, y_query = load_iitd_upd_templates(data_folder, seed=42)

hnsw, leaders = build_hnsw_with_leaders(X_index, k_leaders=5, M=16, ef_construction=200, random_seed=42)

100_L: 3 index, 2 query
100_R: 3 index, 2 query
101_L: 3 index, 2 query
101_R: 3 index, 2 query
102_L: 3 index, 2 query
102_R: 3 index, 2 query
103_L: 3 index, 2 query
103_R: 3 index, 2 query
104_L: 3 index, 2 query
104_R: 3 index, 2 query
105_L: 3 index, 2 query
105_R: 3 index, 2 query
106_L: 3 index, 2 query
106_R: 3 index, 2 query
107_L: 3 index, 2 query
107_R: 3 index, 2 query
108_L: 3 index, 2 query
108_R: 3 index, 2 query
109_L: 3 index, 2 query
109_R: 3 index, 2 query
10_L: 7 index, 3 query
110_L: 3 index, 2 query
110_R: 3 index, 2 query
111_L: 3 index, 2 query
111_R: 3 index, 2 query
112_L: 3 index, 2 query
112_R: 3 index, 2 query
113_L: 3 index, 2 query
113_R: 3 index, 2 query
114_L: 3 index, 2 query
114_R: 3 index, 2 query
115_L: 3 index, 2 query
115_R: 3 index, 2 query
116_L: 3 index, 2 query
116_R: 3 index, 2 query
117_L: 3 index, 2 query
117_R: 3 index, 2 query
118_L: 3 index, 2 query
118_R: 3 index, 2 query
119_L: 3 index, 2 query
119_R: 3 index, 2 query
11_L: 7 index, 3 

In [27]:
results = evaluate_top1_timing(hnsw, y_index, X_query, y_query,ef_search=23)
print("Hit rate:", results["hit_rate"])
print("Average search time (ms):", results["avg_time_s"] * 1000)

Hit rate: 0.9048697621744054
Average search time (ms): 0.49766969771070635


In [9]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query, 50,23)

np.savez('IITDv1_khnsw.npz', x=x_labels, y=hit_rates)

Evaluating top-K hit rates:   2%|▏         | 1/50 [00:00<00:23,  2.10it/s]

K=  1, Hit rate=0.9049


Evaluating top-K hit rates:   4%|▍         | 2/50 [00:00<00:22,  2.09it/s]

K=  2, Hit rate=0.9434


Evaluating top-K hit rates:   6%|▌         | 3/50 [00:01<00:22,  2.10it/s]

...

K= 48, Hit rate=0.9796


Evaluating top-K hit rates:  98%|█████████▊| 49/50 [00:23<00:00,  2.08it/s]

K= 49, Hit rate=0.9796


Evaluating top-K hit rates: 100%|██████████| 50/50 [00:23<00:00,  2.10it/s]

K= 50, Hit rate=0.9796


In [30]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query, 50,23)

    # Plot
plot_hit_rate_plotly_topk(x_labels, hit_rates,23,"IITDV1_K-HNSW (k_leaders={5})")

Evaluating top-K hit rates:   2%|▏         | 1/50 [00:00<00:22,  2.21it/s]

K=  1, Hit rate=0.9049


Evaluating top-K hit rates:   4%|▍         | 2/50 [00:00<00:21,  2.23it/s]

K=  2, Hit rate=0.9434


Evaluating top-K hit rates:   6%|▌         | 3/50 [00:01<00:21,  2.23it/s]

...

K= 48, Hit rate=0.9796


Evaluating top-K hit rates:  98%|█████████▊| 49/50 [00:21<00:00,  2.30it/s]

K= 49, Hit rate=0.9796


Evaluating top-K hit rates: 100%|██████████| 50/50 [00:21<00:00,  2.28it/s]

K= 50, Hit rate=0.9796


In [17]:
start = time.time()   

hnsw, leaders = build_hnsw_with_leaders(X_index, k_leaders=5, M=16, ef_construction=200, random_seed=42)
end = time.time()     

print(f"Time taken: {end - start:.6f} seconds")

Chosen 5 leaders (indices): [164, 454, 227, 983, 40]
Forcing leaders to level 1 (sampled levels example: [0, 1, 0, 0, 0, 0, 0, 0, 0, 1])
Time taken: 2.251506 seconds


In [21]:
start = time.time()   
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query, 50,23)

end = time.time()     

print(f"Time taken: {end - start:.6f} seconds")

K=  1, Hit rate=0.9049
Time taken: 0.434883 seconds


# CASIAV1

In [10]:
def load_CASIAV1_templates(data_folder):
    """
    Load CASIA V1 templates according to the structure:
        <subjectID>_<session>_<sample>.jpg.mat

    Logic:
        - Files ending with '_3.jpg.mat' or '_4.jpg.mat' → Query templates
        - Files ending with '_1.jpg.mat' or '_2.jpg.mat' → Index (enrollment) templates

    Returns:
        index_vectors, index_labels, query_vectors, query_labels
    """
    files = sorted([f for f in os.listdir(data_folder) if f.endswith('.jpg.mat')])
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    for file in files:
        path = os.path.join(data_folder, file)
        data = loadmat(path)
        vec = data['template'].flatten().astype('float32')

        # Extract identifiers
        subject_id, session, sample = file.replace('.jpg.mat', '').split('_')
        label = f"{subject_id}_{session}"  # subject-session pair label

        if sample in ('3', '4'):  # Probe (query)
            query_vectors.append(vec)
            query_labels.append(label)
        else:  # Enrollment (index)
            index_vectors.append(vec)
            index_labels.append(label)

    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors),
        np.array(query_labels)
    )


In [11]:
data_folder = "/home/nishkal/sg/iris_indexing/Updated_CASIAV1"
X_index, y_index, X_query, y_query = load_CASIAV1_templates(data_folder)

hnsw, leaders = build_hnsw_with_leaders(X_index, k_leaders=5, M=8, ef_construction=100, random_seed=42)

Chosen 5 leaders (indices): [136, 411, 24, 373, 11]
Forcing leaders to level 2 (sampled levels example: [0, 1, 0, 0, 0, 0, 0, 1, 0, 1])


In [33]:
results = evaluate_top1_timing(hnsw, y_index, X_query, y_query,ef_search=20)
print("Hit rate:", results["hit_rate"])
print("Average search time (ms):", results["avg_time_s"] * 1000)

Hit rate: 0.558641975308642
Average search time (ms): 1.8979472900096925


In [13]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query, 50,20)

np.savez('CASIAv1_khnsw.npz', x=x_labels, y=hit_rates)

Evaluating top-K hit rates:   2%|▏         | 1/50 [00:00<00:34,  1.44it/s]

K=  1, Hit rate=0.5586


Evaluating top-K hit rates:   4%|▍         | 2/50 [00:01<00:33,  1.41it/s]

K=  2, Hit rate=0.7562


Evaluating top-K hit rates:   6%|▌         | 3/50 [00:02<00:33,  1.41it/s]

...

K= 48, Hit rate=0.9136


Evaluating top-K hit rates:  98%|█████████▊| 49/50 [00:34<00:00,  1.42it/s]

K= 49, Hit rate=0.9136


Evaluating top-K hit rates: 100%|██████████| 50/50 [00:34<00:00,  1.43it/s]

K= 50, Hit rate=0.9136


In [34]:



x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query, 50,20)

    # Plot
plot_hit_rate_plotly_topk(x_labels, hit_rates,20,"CASIAV1_HNSW (k_leaders={5})")

Evaluating top-K hit rates:   2%|▏         | 1/50 [00:00<00:30,  1.61it/s]

K=  1, Hit rate=0.5586


Evaluating top-K hit rates:   4%|▍         | 2/50 [00:01<00:29,  1.63it/s]

K=  2, Hit rate=0.7562


Evaluating top-K hit rates:   6%|▌         | 3/50 [00:01<00:29,  1.61it/s]

...

K= 48, Hit rate=0.9136


Evaluating top-K hit rates:  98%|█████████▊| 49/50 [00:31<00:00,  1.47it/s]

K= 49, Hit rate=0.9136


Evaluating top-K hit rates: 100%|██████████| 50/50 [00:32<00:00,  1.55it/s]

K= 50, Hit rate=0.9136


# IRIS_LAMP

In [15]:

def load_IrisLamp_templates(root_folder, seed=42, pick='random'):
    """
    Process a flat folder of template files named like: 001_L_01_template.npz
    Groups by subject-eye (e.g. '001_L'), picks exactly one file per group for indexing,
    and the rest for querying.

    Args:
        root_folder (str): path to folder containing .npz / .npy template files.
        seed (int): deterministic seed for random picking.
        pick (str): 'random' (default) or 'first' - selection strategy for the 1 index file.

    Returns:
        index_vectors: np.ndarray (M, D) float32
        index_labels:  np.ndarray (M,) strings like '001_L'
        query_vectors: np.ndarray (Q, D) float32 (empty if none)
        query_labels:  np.ndarray (Q,) strings
    """
    random.seed(seed)
    np.random.seed(seed)

    # regex to extract subject and eye from filename; flexible to variations
    # examples matched: 001_L_01_template.npz, 23_R_3_template.npz, 0001_L_12_template.npz
    pattern = re.compile(r'^(\d+)_([LRlr])_.*template', re.IGNORECASE)

    # collect files
    files = [f for f in os.listdir(root_folder) if f.lower().endswith(('.npz', '.npy'))]
    if not files:
        raise ValueError(f"No .npz/.npy files found in {root_folder}")

    groups = {}  # key -> list of filenames
    for f in sorted(files):
        m = pattern.match(f)
        if not m:
            # skip files that don't match pattern (or optionally group them by filename prefix)
            print(f"⚠️ Skipping file with unexpected name format: {f}")
            continue
        subj = m.group(1).zfill(3)  # pad subject id to 3 digits for consistent labels
        eye = m.group(2).upper()
        key = f"{subj}_{eye}"
        groups.setdefault(key, []).append(f)

    # helper to convert a single file into a 1D float32 normalized vector
    def file_to_vector(path):
        full = os.path.join(root_folder, path)
        if full.lower().endswith('.npz'):
            data = np.load(full, allow_pickle=True)
            if 'iris_code' not in data.files or 'mask_code' not in data.files:
                raise ValueError(f"{path} missing 'iris_code' or 'mask_code' keys.")
            iris_code = np.array(data['iris_code']).reshape(-1)
            mask_code = np.array(data['mask_code']).reshape(-1)
            if iris_code.shape != mask_code.shape:
                # try broadcasting or raise
                if iris_code.size == mask_code.size:
                    pass
                else:
                    raise ValueError(f"Shape mismatch in {path}: iris_code {iris_code.shape}, mask_code {mask_code.shape}")
            # mask==1 => occluded/unreliable; set corresponding bits to 0
            vec = np.where(mask_code == 1, 0, iris_code).astype(np.float32)
        elif full.lower().endswith('.npy'):
            vec = np.load(full).reshape(-1).astype(np.float32)
        else:
            raise ValueError(f"Unsupported file: {path}")

        # L2 normalize if possible
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        return vec

    index_vectors = []
    index_labels = []
    query_vectors = []
    query_labels = []

    # iterate groups
    for key in sorted(groups.keys()):
        group_files = groups[key][:]
        if len(group_files) == 0:
            continue

        if pick == 'random':
            random.shuffle(group_files)
        elif pick == 'first':
            group_files = sorted(group_files)
        else:
            raise ValueError("pick must be 'random' or 'first'")

        # pick first as index, rest are query
        idx_file = group_files[0]
        try:
            idx_vec = file_to_vector(idx_file)
        except Exception as e:
            print(f"⚠️ Failed to load index file {idx_file} for {key}: {e}. Skipping this group.")
            continue

        index_vectors.append(idx_vec)
        index_labels.append(key)

        for qf in group_files[1:]:
            try:
                qvec = file_to_vector(qf)
            except Exception as e:
                print(f"⚠️ Failed to load query file {qf} for {key}: {e}. Skipping this file.")
                continue
            query_vectors.append(qvec)
            query_labels.append(key)

        print(f"{key}: index=1, query={max(0, len(group_files)-1)}")

    if len(index_vectors) == 0:
        raise ValueError("No index vectors found. Check filename patterns and folder contents.")

    index_vectors = np.vstack(index_vectors).astype(np.float32)
    index_labels = np.array(index_labels)

    if len(query_vectors) > 0:
        query_vectors = np.vstack(query_vectors).astype(np.float32)
        query_labels = np.array(query_labels)
    else:
        D = index_vectors.shape[1]
        query_vectors = np.zeros((0, D), dtype=np.float32)
        query_labels = np.array([])

    return index_vectors, index_labels, query_vectors, query_labels


In [16]:
data_folder = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Lamp_/outputs_npz/templates"
X_index, y_index, X_query, y_query = load_IrisLamp_templates(data_folder)

hnsw, leaders = build_hnsw_with_leaders(X_index, k_leaders=5, M=16, ef_construction=200, random_seed=42)

001_L: index=1, query=19
001_R: index=1, query=19
002_L: index=1, query=19
002_R: index=1, query=19
003_L: index=1, query=19
003_R: index=1, query=19
004_L: index=1, query=19
004_R: index=1, query=19
005_L: index=1, query=19
005_R: index=1, query=19
006_L: index=1, query=19
006_R: index=1, query=19
007_L: index=1, query=19
007_R: index=1, query=19
008_L: index=1, query=19
008_R: index=1, query=19
009_L: index=1, query=19
009_R: index=1, query=19
010_L: index=1, query=19
010_R: index=1, query=19
011_L: index=1, query=19
011_R: index=1, query=19
012_L: index=1, query=19
012_R: index=1, query=19
013_L: index=1, query=19
013_R: index=1, query=19
014_L: index=1, query=19
014_R: index=1, query=19
015_L: index=1, query=19
015_R: index=1, query=19
016_L: index=1, query=19
016_R: index=1, query=19
017_L: index=1, query=19
017_R: index=1, query=19
018_L: index=1, query=19
018_R: index=1, query=19
019_L: index=1, query=19
019_R: index=1, query=19
020_L: index=1, query=19
020_R: index=1, query=19


In [21]:
results = evaluate_top1_timing(hnsw, y_index, X_query, y_query,ef_search=150)
print("Hit rate:", results["hit_rate"])
print("Average search time (ms):", results["avg_time_s"] * 1000)

Hit rate: 0.8615615028977014
Average search time (ms): 9.886823181889708


In [17]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,100,150)

np.savez('IrisLamp_khnsw.npz', x=x_labels, y=hit_rates)

Evaluating top-K hit rates:   5%|▌         | 1/20 [02:39<50:22, 159.07s/it]

K=  1, Hit rate=0.8616


Evaluating top-K hit rates:  10%|█         | 2/20 [05:17<47:35, 158.65s/it]

K=  6, Hit rate=0.8879


Evaluating top-K hit rates:  15%|█▌        | 3/20 [07:55<44:53, 158.42s/it]

...

K= 86, Hit rate=0.9238


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [52:44<02:40, 160.88s/it]

K= 91, Hit rate=0.9248


Evaluating top-K hit rates: 100%|██████████| 20/20 [55:21<00:00, 166.08s/it]

K= 96, Hit rate=0.9259


In [ ]:



x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,100,150)

    # Plot
plot_hit_rate_plotly_topk(x_labels, hit_rates,150,"IRIS_LAMP HNSW (k_leaders={5})")

001_L: index=1, query=19
001_R: index=1, query=19
002_L: index=1, query=19
002_R: index=1, query=19
003_L: index=1, query=19
003_R: index=1, query=19
004_L: index=1, query=19
004_R: index=1, query=19
005_L: index=1, query=19
005_R: index=1, query=19
006_L: index=1, query=19
006_R: index=1, query=19
007_L: index=1, query=19
007_R: index=1, query=19
008_L: index=1, query=19
008_R: index=1, query=19
009_L: index=1, query=19
009_R: index=1, query=19
010_L: index=1, query=19
010_R: index=1, query=19
011_L: index=1, query=19
011_R: index=1, query=19
012_L: index=1, query=19
012_R: index=1, query=19
013_L: index=1, query=19
013_R: index=1, query=19
014_L: index=1, query=19
014_R: index=1, query=19
015_L: index=1, query=19
015_R: index=1, query=19
016_L: index=1, query=19
016_R: index=1, query=19
017_L: index=1, query=19
017_R: index=1, query=19
018_L: index=1, query=19
018_R: index=1, query=19
019_L: index=1, query=19
019_R: index=1, query=19
020_L: index=1, query=19
020_R: index=1, query=19


Evaluating top-K hit rates:   5%|▌         | 1/20 [02:32<48:23, 152.79s/it]

K=  1, Hit rate=0.8616


Evaluating top-K hit rates:  10%|█         | 2/20 [05:04<45:38, 152.15s/it]

K=  6, Hit rate=0.8879


...

K= 86, Hit rate=0.9238


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [48:11<02:27, 147.31s/it]

K= 91, Hit rate=0.9248


Evaluating top-K hit rates: 100%|██████████| 20/20 [50:35<00:00, 151.79s/it]


K= 96, Hit rate=0.9259


# Iris_Thousand

In [19]:
import os
import re
import random
import numpy as np

def load_IrisThousand_templates(root_folder, seed=42, pick='random', expected_per_group=10):
    """
    Loader for 'Iris Thousand' style filenames like:
      000_L_00_template.npz ... 000_L_09_template.npz
      000_R_00_template.npz ... 000_R_09_template.npz
      ...
      999_R_09_template.npz

    Behavior:
      - Groups files by subject (0..999) and eye (L/R).
      - Picks exactly one file per group as the index vector (strategy: 'random' or 'first').
      - All remaining files in that group become queries for that label.

    Args:
      root_folder (str): path to folder containing .npz / .npy template files.
      seed (int): random seed for deterministic shuffling.
      pick (str): 'random' (default) or 'first' - selection strategy for the 1 index file.
      expected_per_group (int): expected images per subject-eye (default 10). Used to warn if mismatch.

    Returns:
      index_vectors: np.ndarray (M, D) float32
      index_labels:  np.ndarray (M,) strings like '000_L'
      query_vectors: np.ndarray (Q, D) float32 (empty if none)
      query_labels:  np.ndarray (Q,) strings
    """
    random.seed(seed)
    np.random.seed(seed)

    # Pattern extracts subject (1-3 digits), eye (L/R) and image number (00..09 or similar)
    pattern = re.compile(r'^(\d{1,3})_([LRlr])_0*(\d+)_template', re.IGNORECASE)

    files = [f for f in os.listdir(root_folder) if f.lower().endswith(('.npz', '.npy'))]
    if not files:
        raise ValueError(f"No .npz/.npy files found in {root_folder}")

    groups = {}
    for f in sorted(files):
        m = pattern.match(f)
        if not m:
            print(f"⚠️ Skipping file with unexpected name format: {f}")
            continue
        subj = m.group(1).zfill(3)   # ensure three-digit subject id like '000'
        eye = m.group(2).upper()
        key = f"{subj}_{eye}"
        groups.setdefault(key, []).append(f)

    def file_to_vector(path):
        full = os.path.join(root_folder, path)
        if full.lower().endswith('.npz'):
            data = np.load(full, allow_pickle=True)
            # keep the iris_code / mask_code logic you already used
            if 'iris_code' not in data.files or 'mask_code' not in data.files:
                raise ValueError(f"{path} missing 'iris_code' or 'mask_code' keys.")
            iris_code = np.array(data['iris_code']).reshape(-1)
            mask_code = np.array(data['mask_code']).reshape(-1)
            if iris_code.shape != mask_code.shape:
                if iris_code.size == mask_code.size:
                    pass
                else:
                    raise ValueError(f"Shape mismatch in {path}: iris_code {iris_code.shape}, mask_code {mask_code.shape}")
            vec = np.where(mask_code == 1, 0, iris_code).astype(np.float32)
        elif full.lower().endswith('.npy'):
            vec = np.load(full).reshape(-1).astype(np.float32)
        else:
            raise ValueError(f"Unsupported file: {path}")

        # L2 normalize
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        return vec

    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    for key in sorted(groups.keys()):
        group_files = groups[key][:]
        if len(group_files) == 0:
            continue

        if len(group_files) != expected_per_group:
            print(f"⚠️ Warning: group {key} has {len(group_files)} files (expected {expected_per_group}).")

        if pick == 'random':
            random.shuffle(group_files)
        elif pick == 'first':
            group_files = sorted(group_files)
        else:
            raise ValueError("pick must be 'random' or 'first'")

        idx_file = group_files[0]
        try:
            idx_vec = file_to_vector(idx_file)
        except Exception as e:
            print(f"⚠️ Failed to load index file {idx_file} for {key}: {e}. Skipping this group.")
            continue

        index_vectors.append(idx_vec)
        index_labels.append(key)

        for qf in group_files[1:]:
            try:
                qvec = file_to_vector(qf)
            except Exception as e:
                print(f"⚠️ Failed to load query file {qf} for {key}: {e}. Skipping this file.")
                continue
            query_vectors.append(qvec)
            query_labels.append(key)

        print(f"{key}: index=1, query={max(0, len(group_files)-1)}")

    if len(index_vectors) == 0:
        raise ValueError("No index vectors found. Check filename patterns and folder contents.")

    index_vectors = np.vstack(index_vectors).astype(np.float32)
    index_labels = np.array(index_labels)

    if len(query_vectors) > 0:
        query_vectors = np.vstack(query_vectors).astype(np.float32)
        query_labels = np.array(query_labels)
    else:
        D = index_vectors.shape[1]
        query_vectors = np.zeros((0, D), dtype=np.float32)
        query_labels = np.array([])

    return index_vectors, index_labels, query_vectors, query_labels


In [20]:
data_folder = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Thousand_/outputs_npz/templates"
X_index, y_index, X_query, y_query = load_IrisThousand_templates(data_folder)

hnsw, leaders = build_hnsw_with_leaders(X_index, k_leaders=7, M=16, ef_construction=200, random_seed=42)




⚠️ Warning: group 000_L has 9 files (expected 10).
000_L: index=1, query=8
000_R: index=1, query=9
001_L: index=1, query=9
⚠️ Warning: group 001_R has 9 files (expected 10).
001_R: index=1, query=8
002_L: index=1, query=9
⚠️ Warning: group 002_R has 8 files (expected 10).
002_R: index=1, query=7
003_L: index=1, query=9
003_R: index=1, query=9
004_L: index=1, query=9
004_R: index=1, query=9
005_L: index=1, query=9
005_R: index=1, query=9
⚠️ Warning: group 006_L has 7 files (expected 10).
006_L: index=1, query=6
006_R: index=1, query=9
007_L: index=1, query=9
⚠️ Warning: group 007_R has 7 files (expected 10).
007_R: index=1, query=6
008_L: index=1, query=9
008_R: index=1, query=9
009_L: index=1, query=9
009_R: index=1, query=9
⚠️ Warning: group 010_L has 8 files (expected 10).
010_L: index=1, query=7
⚠️ Warning: group 010_R has 7 files (expected 10).
010_R: index=1, query=6
011_L: index=1, query=9
011_R: index=1, query=9
⚠️ Warning: group 012_L has 9 files (expected 10).
012_L: index=1, 

In [13]:
root = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Thousand_/outputs_npz/templates"
idx_vecs, idx_labels, qry_vecs, qry_labels = load_IrisThousand_templates(root, seed=123, pick='random')
print("Index:", idx_vecs.shape, idx_labels.shape)
print("Query:", qry_vecs.shape, qry_labels.shape)

⚠️ Warning: group 000_L has 9 files (expected 10).
000_L: index=1, query=8
000_R: index=1, query=9
001_L: index=1, query=9
⚠️ Warning: group 001_R has 9 files (expected 10).
001_R: index=1, query=8
002_L: index=1, query=9
⚠️ Warning: group 002_R has 8 files (expected 10).
002_R: index=1, query=7
003_L: index=1, query=9
003_R: index=1, query=9
004_L: index=1, query=9
004_R: index=1, query=9
005_L: index=1, query=9
005_R: index=1, query=9
⚠️ Warning: group 006_L has 7 files (expected 10).
006_L: index=1, query=6
006_R: index=1, query=9
007_L: index=1, query=9
⚠️ Warning: group 007_R has 7 files (expected 10).
007_R: index=1, query=6
008_L: index=1, query=9
008_R: index=1, query=9
009_L: index=1, query=9
009_R: index=1, query=9
⚠️ Warning: group 010_L has 8 files (expected 10).
010_L: index=1, query=7
⚠️ Warning: group 010_R has 7 files (expected 10).
010_R: index=1, query=6
011_L: index=1, query=9
011_R: index=1, query=9
⚠️ Warning: group 012_L has 9 files (expected 10).
012_L: index=1, 

In [21]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,200,200)

np.savez('IrisThousand_khnsw.npz', x=x_labels, y=hit_rates)

Evaluating top-K hit rates:   5%|▌         | 1/20 [05:07<1:37:24, 307.59s/it]

K=  1, Hit rate=0.6159


Evaluating top-K hit rates:  10%|█         | 2/20 [10:12<1:31:52, 306.26s/it]

K= 11, Hit rate=0.6728


Evaluating top-K hit rates:  15%|█▌        | 3/20 [15:18<1:26:41, 305.95s/it]

...

K=171, Hit rate=0.7474


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [1:36:50<05:05, 305.72s/it]

K=181, Hit rate=0.7489


Evaluating top-K hit rates: 100%|██████████| 20/20 [1:41:44<00:00, 305.22s/it]

K=191, Hit rate=0.7520


In [ ]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,100,200)
# Plot
plot_hit_rate_plotly_topk(x_labels, hit_rates,150,"IRIS_Thousand K-HNSW (k_leaders={7})")

Evaluating top-K hit rates:   5%|▌         | 1/20 [05:12<1:39:06, 312.98s/it]

K=  1, Hit rate=0.6159


Evaluating top-K hit rates:  10%|█         | 2/20 [10:29<1:34:28, 314.89s/it]

K=  6, Hit rate=0.6586


Evaluating top-K hit rates:  15%|█▌        | 3/20 [15:41<1:28:52, 313.65s/it]

...

K= 86, Hit rate=0.7253


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [1:38:30<05:09, 309.92s/it]

K= 91, Hit rate=0.7271


Evaluating top-K hit rates: 100%|██████████| 20/20 [1:43:30<00:00, 310.54s/it]

K= 96, Hit rate=0.7291


In [16]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,200,200)
# Plot
plot_hit_rate_plotly_topk(x_labels, hit_rates,150,"IRIS_Thousand K-HNSW (k_leaders={7})")

Evaluating top-K hit rates:   5%|▌         | 1/20 [05:06<1:37:12, 306.95s/it]

K=  1, Hit rate=0.6159


Evaluating top-K hit rates:  10%|█         | 2/20 [10:14<1:32:15, 307.53s/it]

K= 11, Hit rate=0.6728


Evaluating top-K hit rates:  15%|█▌        | 3/20 [15:19<1:26:48, 306.38s/it]

...

K=171, Hit rate=0.7474


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [1:36:53<05:06, 306.35s/it]

K=181, Hit rate=0.7489


Evaluating top-K hit rates: 100%|██████████| 20/20 [1:41:59<00:00, 305.96s/it]

K=191, Hit rate=0.7520


In [18]:
results = evaluate_top1_timing(hnsw, y_index, X_query, y_query,ef_search=200)
print("Hit rate:", results["hit_rate"])
print("Average search time (ms):", results["avg_time_s"] * 1000)

Hit rate: 0.6159398313823012
Average search time (ms): 18.257792464611608
